# Limpieza de Datos de Establecimientos Educativos (Diversificado)
**Integrantes:** 
- Sofía García
- Julio García Salas
- Joaquin Campos 

## Paso 2: Explorar el estado de los datos

- ¿Qué columnas tenemos?
- ¿Cuáles parecen ser clave?
- ¿Hay valores nulos?
- ¿Columnas redundantes o mal nombradas?
- Valores de las columnas 


In [13]:
import pandas as pd
import re
# Cargar datos unificados
df = pd.read_csv('todos_los_establecimientos.csv', encoding='utf-8-sig')

# Dimensiones del dataset
print(f"Filas: {df.shape[0]:,}, Columnas: {df.shape[1]}")

# Ver las primeras columnas y filas
df.head()


Filas: 16,414, Columnas: 17


,CODIGO,DISTRITO,DEPARTAMENTO,MUNICIPIO,ESTABLECIMIENTO,DIRECCION,TELEFONO,SUPERVISOR,DIRECTOR,NIVEL,SECTOR,AREA,STATUS,MODALIDAD,JORNADA,PLAN,DEPARTAMENTAL
0,16-01-0026-45,16-031,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO IMPERIAL,5A. CALLE 1-98 ZONA 3,57101061,PATRICIO NAJARRO ASENCIO,MYNOR GUSTAVO IPIÑA ESPAÑA,BASICO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,DOBLE,FIN DE SEMANA,ALTA VERAPAZ
1,16-01-0135-45,16-005,ALTA VERAPAZ,COBAN,INEB ADSCRITO A INSTITUTO 'EMILIO ROSALES PONCE',3A AVE 6-23 ZONA 11,79529782,NORA LILIANA FIGUEROA HERNÁNDEZ,VICTOR HUGO DOMÍNGUEZ REYES,BASICO,OFICIAL,URBANA,ABIERTA,BILINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
2,16-01-0136-45,16-005,ALTA VERAPAZ,COBAN,INEB,6A AVE 1-15 ZONA 4,79513568,NORA LILIANA FIGUEROA HERNÁNDEZ,WUENDY LUCRECIA ESTRADA BEDOYA,BASICO,OFICIAL,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
3,16-01-0138-45,16-031,ALTA VERAPAZ,COBAN,COLEGIO COBAN,"KM.2 SALIDA A SAN JUAN CHAMELCO, ZONA 8",77945104,PATRICIO NAJARRO ASENCIO,GUSTAVO ADOLFO SIERRA POP,BASICO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ
4,16-01-0139-45,16-031,ALTA VERAPAZ,COBAN,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,77367402,PATRICIO NAJARRO ASENCIO,GILMA DOLORES GUAY PAZ DE LEAL,BASICO,PRIVADO,URBANA,ABIERTA,MONOLINGUE,MATUTINA,DIARIO(REGULAR),ALTA VERAPAZ


In [14]:
df.columns.tolist()


['CODIGO',
 'DISTRITO',
 'DEPARTAMENTO',
 'MUNICIPIO',
 'ESTABLECIMIENTO',
 'DIRECCION',
 'TELEFONO',
 'SUPERVISOR',
 'DIRECTOR',
 'NIVEL',
 'SECTOR',
 'AREA',
 'STATUS',
 'MODALIDAD',
 'JORNADA',
 'PLAN',
 'DEPARTAMENTAL']

In [15]:
df.isna().sum().sort_values(ascending=False)


TELEFONO           424
DIRECTOR            63
DIRECCION           10
SECTOR               0
PLAN                 0
JORNADA              0
MODALIDAD            0
STATUS               0
AREA                 0
CODIGO               0
NIVEL                0
DISTRITO             0
SUPERVISOR           0
ESTABLECIMIENTO      0
MUNICIPIO            0
DEPARTAMENTO         0
DEPARTAMENTAL        0
dtype: int64

La mayoría de los campos están completos y bien distribuidos.

Solo hay algunos casos donde faltan teléfonos y en menor medida nombres de directores o direcciones.


## Paso 1: Normalización de texto

Objetivo: Hacer que todos los textos estén en un formato uniforme, eliminando espacios extra y asegurando que todo esté en mayúsculas para facilitar comparaciones y análisis posteriores.

Columnas a limpiar:
- `ESTABLECIMIENTO`
- `DIRECCION`
- `TELEFONO`
- `SUPERVISOR`
- `DIRECTOR`


In [16]:
# Copia del dataframe original por seguridad
df_limpio = df.copy()

# Lista de columnas de texto a normalizar
columnas_texto = ['ESTABLECIMIENTO', 'DIRECCION', 'TELEFONO', 'SUPERVISOR', 'DIRECTOR']

for col in columnas_texto:
    df_limpio[col] = (
        df_limpio[col]
        .fillna('')                       # Rellena vacíos con cadena vacía
        .astype(str)                      # Asegura que todos sean strings
        .str.replace('\xa0', ' ', regex=False)  # NBSP → espacio
        .apply(lambda x: ' '.join(x.split()))   # Quita espacios extra
        .str.upper()                     # Convierte a mayúsculas
    )
    


## Estado de los datos

- **Datos completos** en la mayoría de las columnas, con valores faltantes solo en:
  - `TELEFONO` (424 casos)
  - `DIRECTOR` (63 casos)
  - `DIRECCION` (10 casos)
- Los campos de texto presentan inconsistencias en el uso de mayúsculas/minúsculas y espacios extra.
- En la columna `TELEFONO` aparecen valores con el formato `79416669.0` debido a que fueron interpretados como números flotantes (`float`) durante la carga del CSV.
- En algunos casos, debemos suponer y tratar que al convertir un número a `float`, se perdió un cero inicial, lo que reduce la longitud del número a 7 dígitos.
- Existen teléfonos vacíos que deben marcarse como `"NO DISPONIBLE"`.
- En general, el formato de los textos no es uniforme (acentos, tildes, caracteres invisibles como `\xa0`).

---

## Operaciones de limpieza a realizar

1. **Normalización de teléfonos**:
   - Quitar el `.0` al final de los valores numéricos.
   - Si el número tiene 7 dígitos, agregar un `0` al inicio para corregir la pérdida del cero inicial.
   - Si la longitud es 0 (vacío), reemplazar por `"NO DISPONIBLE"`.

2. **Estandarización de texto**:
   - Convertir a mayúsculas para evitar diferencias por capitalización.
   - Eliminar espacios al inicio y final, así como espacios repetidos entre palabras.
   - Sustituir caracteres invisibles (como `\xa0`) por espacios.

3. **Unificación de formato**:
   - En campos de texto como `ESTABLECIMIENTO`, `DIRECCION`, `SUPERVISOR` y `DIRECTOR`, aplicar la misma normalización para facilitar la detección de duplicados o errores tipográficos.

4. **Revisión de duplicados**:
   - Detectar registros con nombre de establecimiento y dirección idénticos o muy similares.

---


In [17]:
df_limpio = df.copy()

def limpiar_telefono(valor):
    """
    Limpia y normaliza los números de teléfono:
    1. Convierte a string y elimina espacios extra.
    2. Quita '.0' al final si existe.
    3. Si la longitud es 7 dígitos, agrega un cero al inicio.
    4. Si está vacío, devuelve 'NO DISPONIBLE'.
    """
    
    # Si el valor NaN
    if pd.isna(valor):
        return "NO DISPONIBLE"
    
    tel = str(valor).strip()
    tel = tel.replace('\xa0', ' ')
    
    if tel.endswith(".0"):
        tel = tel[:-2]
    tel = "".join(tel.split())
    
    # Si está vacío después de limpiar, marcamos como NO DISPONIBLE
    if len(tel) == 0:
        return "NO DISPONIBLE"
    
    # Si solo hay dígitos y la longitud es 7, agregamos un cero inicial
    if tel.isdigit() and len(tel) == 7:
        tel = "0" + tel
    
    return tel

df_limpio['TELEFONO'] = df_limpio['TELEFONO'].apply(limpiar_telefono)

df_limpio['TELEFONO'].sample(10)


11316    54803826
8825     57006014
8470     46355997
10215    53395878
8768     79305318
10592    57276870
303      53221451
12320    77716637
15614    56233638
815      30358725
Name: TELEFONO, dtype: object

---
## Normalizacion de textos:
Evitar espacios dobles, y en los extremos. porque así normalizamos, además también definimos una "AVENIDA" porque evitamos problemas y seguimos 
normalizando
- AV. / AV → AVENIDA

- Z. / ZN → ZONA

---

In [18]:
cols_texto = ['ESTABLECIMIENTO', 'DIRECCION', 'SUPERVISOR', 'DIRECTOR']
cols_texto = [c for c in cols_texto if c in df_limpio.columns]  

def normalizar_texto(s: pd.Series) -> pd.Series:
    s = s.fillna('').astype(str)
    s = (s
         .str.replace('\xa0', ' ', regex=False)  
         .apply(lambda x: ' '.join(x.split()))  
         .str.upper()                            
    )
    return s

def normalizar_direccion(s: pd.Series) -> pd.Series:
    s = normalizar_texto(s)
    reemplazos = [
        (r'\bAV\.\b', 'AVENIDA'),
        (r'\bAV\b', 'AVENIDA'),
        (r'\bZN\b', 'ZONA'),
        (r'\bZ\.\b', 'ZONA'),
    ]
    for patron, repl in reemplazos:
        s = s.str.replace(patron, repl, regex=True)
    s = s.apply(lambda x: ' '.join(x.split())) 
    return s

# Aplicamos y contamos cambios
for col in cols_texto:
    original = df_limpio[col].copy()
    
    if col == 'DIRECCION':
        df_limpio[col] = normalizar_direccion(df_limpio[col])
    else:
        df_limpio[col] = normalizar_texto(df_limpio[col])
    
    cambios = (original != df_limpio[col]).sum()
    print(f"Columna '{col}': {cambios} registros modificados.")

df_limpio[cols_texto].head(10)


Columna 'ESTABLECIMIENTO': 0 registros modificados.
Columna 'DIRECCION': 386 registros modificados.
Columna 'SUPERVISOR': 0 registros modificados.
Columna 'DIRECTOR': 63 registros modificados.


,ESTABLECIMIENTO,DIRECCION,SUPERVISOR,DIRECTOR
0,COLEGIO PARTICULAR MIXTO IMPERIAL,5A. CALLE 1-98 ZONA 3,PATRICIO NAJARRO ASENCIO,MYNOR GUSTAVO IPIÑA ESPAÑA
1,INEB ADSCRITO A INSTITUTO 'EMILIO ROSALES PONCE',3A AVE 6-23 ZONA 11,NORA LILIANA FIGUEROA HERNÁNDEZ,VICTOR HUGO DOMÍNGUEZ REYES
2,INEB,6A AVE 1-15 ZONA 4,NORA LILIANA FIGUEROA HERNÁNDEZ,WUENDY LUCRECIA ESTRADA BEDOYA
3,COLEGIO COBAN,"KM.2 SALIDA A SAN JUAN CHAMELCO, ZONA 8",PATRICIO NAJARRO ASENCIO,GUSTAVO ADOLFO SIERRA POP
4,COLEGIO PARTICULAR MIXTO VERAPAZ,KM 209.5 ENTRADA A LA CIUDAD,PATRICIO NAJARRO ASENCIO,GILMA DOLORES GUAY PAZ DE LEAL
5,"COLEGIO ""LA INMACULADA""",7A. AVENIDA 11-109 ZONA 6,PATRICIO NAJARRO ASENCIO,VIRGINIA SOLANO SERRANO
6,INSTITUTO NACIONAL DE EDUCACION BASICA DE TELE...,ALDEA SAMOX SAN LUCAS,JOSE ARTURO CHOC CHEN,DEBORA ESMERALDA NATARENO FLORES
7,INSTITUTO NACIONAL DE EDUCACION BASICA DE TELE...,ALDEA CAMCAL,JOSE ARTURO CHOC CHEN,DOMINGO TOT COY
8,"LICEO ""MODERNO LATINO""",11 AVENIDA 5-17 ZONA 4,PATRICIO NAJARRO ASENCIO,HÉCTOR ARMANDO TEYUL CHEN
9,COLEGIO PRIVADO MIXTO TECNOLÓGICO EN INFORMÁTICA,"2A. CALLE 12-23, ZONA 4",PATRICIO NAJARRO ASENCIO,JORGE SALVADOR JUÁREZ SIERRA


---
## Tipos de dato seguros:
Para evitar que CODIGO, códigos distritales o campos de texto se conviertan en números y pierdan formato.



---

In [19]:
import unicodedata

cols_a_texto = [
    'CODIGO','DISTRITO','DEPARTAMENTO','MUNICIPIO','ESTABLECIMIENTO','DIRECCION',
    'TELEFONO','SUPERVISOR','DIRECTOR','NIVEL','SECTOR','AREA','STATUS',
    'MODALIDAD','JORNADA','PLAN','DEPARTAMENTAL'
]
cols_a_texto = [c for c in cols_a_texto if c in df_limpio.columns]

cambios_tipo = {}
for col in cols_a_texto:
    antes = df_limpio[col].copy()
    # a texto y quitar sufijo .0 solo si aparece al final
    df_limpio[col] = (
        df_limpio[col]
        .astype(str)
        .str.replace('\xa0',' ', regex=False)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )
    cambios_tipo[col] = (antes.astype(str) != df_limpio[col]).sum()

print("Limpieza NROM — Cambios por normalización de tipo:")
for k,v in cambios_tipo.items():
    if v>0:
        print(f"  {k}: {v} valores ajustados")

Limpieza NROM — Cambios por normalización de tipo:


---
## Estandarización Categórica

Para evitar variantes de escritura (matutina/MATUTINO, vespertino/vespertina, etc.).


---

In [20]:
def normaliza_texto_basico(s: pd.Series) -> pd.Series:
    s = s.fillna('').astype(str)
    s = (s
         .str.replace('\xa0',' ', regex=False)
         .apply(lambda x: ' '.join(x.split()))
         .str.upper()
    )
    return s

# Mapas conservadores (solo alias evidentes)
map_jornada = {
    'MATUTINO': 'MATUTINA', 'MAT': 'MATUTINA',
    'VESPERTINO': 'VESPERTINA', 'VESP': 'VESPERTINA',
    'NOCTURNO': 'NOCTURNA', 'NOC': 'NOCTURNA',
    'DOBLE JORNADA': 'DOBLE JORNADA'
}
map_sector = {
    'PUBLICO': 'OFICIAL', 'PÚBLICO': 'OFICIAL',
    'PRIVADA': 'PRIVADO'
}
map_area = {
    'URBANO': 'URBANA', 'RURAL': 'RURAL'
}
map_status = {
    'ACTIVA': 'ACTIVO', 'INACTIVA': 'INACTIVO'
}

def aplicar_mapa(col, mapa):
    original = df_limpio[col].copy()
    df_limpio[col] = normaliza_texto_basico(df_limpio[col]).replace(mapa)
    cambios = (original != df_limpio[col]).sum()
    print(f"Limpieza vals — '{col}': {cambios} valores estandarizados.")

for col, mapa in [('JORNADA', map_jornada),
                  ('SECTOR', map_sector),
                  ('AREA', map_area),
                  ('STATUS', map_status)]:
    if col in df_limpio.columns:
        aplicar_mapa(col, mapa)


def normalizar_direccion_2(s: pd.Series) -> pd.Series:
    s = normaliza_texto_basico(s)
    # Reemplazos seguros y frecuentes
    reemplazos = [
        (r'\bAV\.\b', 'AVENIDA'),
        (r'\bAV\b', 'AVENIDA'),
        (r'\bCALZ\.\b', 'CALZADA'),
        (r'\bCARR\.\b', 'CARRETERA'),
        (r'\bZN\b', 'ZONA'),
        (r'\bZ\.\b', 'ZONA')
    ]
    for patron, repl in reemplazos:
        s = s.str.replace(patron, repl, regex=True)
    s = s.apply(lambda x: ' '.join(x.split()))
    return s

if 'DIRECCION' in df_limpio.columns:
    antes = df_limpio['DIRECCION'].copy()
    df_limpio['DIRECCION'] = normalizar_direccion_2(df_limpio['DIRECCION'])
    cambios = (antes != df_limpio['DIRECCION']).sum()
    print(f"Limpieza DIRS — 'DIRECCION': {cambios} valores normalizados.")


Limpieza vals — 'JORNADA': 0 valores estandarizados.
Limpieza vals — 'SECTOR': 0 valores estandarizados.
Limpieza vals — 'AREA': 0 valores estandarizados.
Limpieza vals — 'STATUS': 0 valores estandarizados.
Limpieza DIRS — 'DIRECCION': 0 valores normalizados.


---
## Marca de posibles duplicados

Porque ayudar a detectar registros repetidos sin eliminar filas.



---


In [21]:
def quitar_acentos(texto: str) -> str:
    texto = unicodedata.normalize('NFD', texto)
    texto = ''.join(ch for ch in texto if unicodedata.category(ch) != 'Mn')
    return texto

def canonizar_cadena(texto: str) -> str:
    t = texto.upper()
    t = quitar_acentos(t)
    t = re.sub(r'[^A-Z0-9\s]', ' ', t)  
    t = ' '.join(t.split())             
    return t

for col in ['ESTABLECIMIENTO','MUNICIPIO','DIRECCION']:
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio[col].fillna('').astype(str)

if all(c in df_limpio.columns for c in ['ESTABLECIMIENTO','MUNICIPIO','DIRECCION']):
    llave = (df_limpio['ESTABLECIMIENTO'] + ' | ' +
             df_limpio['MUNICIPIO'] + ' | ' +
             df_limpio['DIRECCION']).apply(canonizar_cadena)
    df_limpio['LLAVE_CANONICA'] = llave

    conteos = df_limpio['LLAVE_CANONICA'].value_counts()
    df_limpio['POSIBLE_DUPLICADO'] = df_limpio['LLAVE_CANONICA'].map(lambda x: conteos.get(x,0) > 1)

    print("Limpieza DUPS — Posibles duplicados encontrados:",
          int(df_limpio['POSIBLE_DUPLICADO'].sum()))

if 'TELEFONO' in df_limpio.columns:
    antes = df_limpio['TELEFONO'].copy()

    def formatear_tel(t):
        if t == "NO DISPONIBLE":
            return t
        solo_digitos = re.sub(r'\D', '', str(t))
        if len(solo_digitos) == 8:
            return solo_digitos[:4] + '-' + solo_digitos[4:]
        return t  # dejamos tal cual si no cumple

    df_limpio['TELEFONO'] = df_limpio['TELEFONO'].apply(formatear_tel)
    cambios = (antes != df_limpio['TELEFONO']).sum()
    print(f"Limpieza TELS — 'TELEFONO': {cambios} formateados como ####-####.")


Limpieza DUPS — Posibles duplicados encontrados: 7529
Limpieza TELS — 'TELEFONO': 15902 formateados como ####-####.


---
## Valores extraños y consistencias 

Para alertar si hay etiquetas fuera de lo común y  para detectar posibles incongruencias municipio–departamento

---

In [30]:

cat_cols = ['JORNADA','SECTOR','AREA','STATUS','MODALIDAD','NIVEL']
cat_cols = [c for c in cat_cols if c in df_limpio.columns]

# Listas conservadoras (solo donde hay consenso claro)
esperados = {
    'JORNADA': {'MATUTINA','VESPERTINA','NOCTURNA','DOBLE', 'SIN JORNADA', 'INTERMEDIA'},
    'SECTOR': {'OFICIAL','PRIVADO','COOPERATIVA', 'MUNICIPAL'},
    'AREA': {'URBANA','RURAL','SIN ESPECIFICAR'},
    'STATUS': {'ABIERTA', ''},
    'MODALIDAD':{'MONOLINGUE', 'BILINGUE'}
}

print("== Revisión de categóricos ==")
for col in cat_cols:
    valores = sorted(df_limpio[col].dropna().unique().tolist())
    print(f"\n{col}: {len(valores)} valores únicos")
    print("Ejemplos:", valores[:15])  # muestra primeros 15

    if col in esperados:
        fuera = [v for v in valores if v not in esperados[col]]
        if fuera:
            print(f"  *Valores NO esperados en {col}*:", fuera)
        else:
            print(f"  Todos los valores de {col} están dentro de lo esperado.")


== Revisión de categóricos ==

JORNADA: 6 valores únicos
Ejemplos: ['DOBLE', 'INTERMEDIA', 'MATUTINA', 'NOCTURNA', 'SIN JORNADA', 'VESPERTINA']
  Todos los valores de JORNADA están dentro de lo esperado.

SECTOR: 4 valores únicos
Ejemplos: ['COOPERATIVA', 'MUNICIPAL', 'OFICIAL', 'PRIVADO']
  Todos los valores de SECTOR están dentro de lo esperado.

AREA: 3 valores únicos
Ejemplos: ['RURAL', 'SIN ESPECIFICAR', 'URBANA']
  Todos los valores de AREA están dentro de lo esperado.

STATUS: 1 valores únicos
Ejemplos: ['ABIERTA']
  Todos los valores de STATUS están dentro de lo esperado.

MODALIDAD: 2 valores únicos
Ejemplos: ['BILINGUE', 'MONOLINGUE']
  Todos los valores de MODALIDAD están dentro de lo esperado.

NIVEL: 2 valores únicos
Ejemplos: ['BASICO', 'DIVERSIFICADO']


---
## Columnas vacias y Duplicados exactos

Para ver si alguna columna quedó sin datos y para saber si hay filas idénticas

---

In [23]:
cols_vacias = []
for col in df_limpio.columns:
    vacia = df_limpio[col].replace('', pd.NA).isna().all()
    if vacia:
        cols_vacias.append(col)

print("\n== Columnas 100% vacías ==")
print(cols_vacias if cols_vacias else "Ninguna columna está completamente vacía.")


dup_mask = df_limpio.duplicated(keep=False)
num_dups = int(dup_mask.sum())
print("\n== Duplicados exactos (fila completa) ==")
print(f"Filas que aparecen duplicadas exactamente: {num_dups}")

# Si quieres ver una pequeña muestra de duplicados:
if num_dups > 0:
    print("Ejemplo de índices duplicados:", df_limpio[dup_mask].index[:10].tolist())



== Columnas 100% vacías ==
Ninguna columna está completamente vacía.

== Duplicados exactos (fila completa) ==
Filas que aparecen duplicadas exactamente: 0


---
## Duplicidad de establecimiento  

(Inciso 5)
---